# 82514 · Sesión S19 — Filtro de partículas y localización de Monte Carlo

**Bloque 6** · lunes 23 de noviembre de 2026 · 2 h  ·  IQS Universitat Ramon Llull

**Qué hace este cuaderno.** El robot del almacén del curso no sabe dónde empieza. La creencia se representa con una nube de partículas que se mueven con la odometría, se pesan con el sensor y se remuestrean: primero en el pasillo de S18, con las mismas medidas que vio el histograma; después en el almacén entero, con el LiDAR de S6 y un campo de verosimilitud. Se mide cuántas partículas hacen falta (la tasa de éxito en N arranques), qué le pasa al EKF de S18 cuando empieza en el hueco equivocado y cómo se recupera la nube de un secuestro.

**Se apoya en:** Thrun, Burgard y Fox (2005), caps. 4, 6 y 8; Fox (2003); Corke (2023), cap. 6; documentación de numpy y scipy.

**Cómo usarlo en clase.** Sigue el guion de la sesión S19 en las diapositivas del bloque 6. Ejecuta la primera celda una sola vez al empezar; en Colab tarda unos segundos. Las celdas marcadas **Ejercicio** son para que los trabajen los estudiantes: las soluciones están al final del cuaderno.

---

In [ ]:
# Ejecutar una sola vez al abrir el cuaderno. En Colab tarda unos segundos: no instala nada.
import sys, time
import numpy as np, scipy, matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from matplotlib.colors import ListedColormap
from scipy.ndimage import distance_transform_edt

FIJAS = {'numpy': '2.4.6', 'scipy': '1.16.3', 'matplotlib': '3.10.0'}   # las de Colab con las que se probó el bloque 6
EN_COLAB = 'google.colab' in sys.modules
np.set_printoptions(precision=3, suppress=True)
plt.rcParams['figure.figsize'] = (10, 3.6)
IQS_AZUL, IQS_VERDE, GRIS, AMBAR, ROJO = '#1B2A80', '#1FA355', '#8A8FA8', '#B9770E', '#C0392B'
COMA = FuncFormatter(lambda v, p: f'{v:g}'.replace('.', ',').replace('-', '−'))   # ejes con coma decimal

def coma(x, n=2):
    """Un número con coma decimal y signo menos, para imprimir."""
    return f'{x:.{n}f}'.replace('.', ',').replace('-', '−')

def ejes(*axs):
    for ax in axs:
        ax.xaxis.set_major_formatter(COMA); ax.yaxis.set_major_formatter(COMA)

for nombre, mod in (('numpy', np), ('scipy', scipy), ('matplotlib', matplotlib)):
    print(nombre, mod.__version__, '' if mod.__version__ == FIJAS[nombre] else f'(probado con {FIJAS[nombre]})')
print('en Colab:', EN_COLAB)
if not np.__version__.startswith('2.4'):
    print('AVISO: este cuaderno se probó con numpy 2.4; las cifras con semilla pueden cambiar un poco.')

> **Aviso para Colab.** Ábrelo al empezar la sesión y ejecuta la primera celda: no instala nada (usa numpy, scipy y matplotlib de Colab) y tarda unos segundos.
>
> **En clase (seis minutos y medio):** las secciones 0, 2 y 3 sin detenerte, y el taller, las secciones 4, 5 y 6, que se ponen en común al acabar. Después, abre el cuaderno S19b y ejecuta su primera celda.
>
> **En casa:** la sección 1 (el pasillo y las cinco partículas, paso a paso) y los ejercicios 1 a 3; las soluciones están al final.
>
> **No se sube:** este cuaderno no tiene tarea en Moodle.

## 0. El almacén, el robot y el LiDAR

*En clase, ejecuta esta celda sin detenerte.*

Es el simulador del cuaderno S18b, el mismo almacén de S22 (13 × 9 m, celdas de 10 cm, dos naves y un muro con una puerta ancha y una estrecha), el mismo robot (la rueda de S6, 0,5 m/s, odometría a 10 Hz con 6 cm y 0,9° de error por metro) y las mismas diez balizas. Le añade lo de hoy:

- **el pasillo de S18**, con su detector de huecos (0,8 y 0,1);
- **el LiDAR de S6** en la rejilla: 1440 haces de 0,25°, 2 cm de ruido, un 2 % de rayos sin retorno y 10 m de alcance;
- **el campo de verosimilitud** y **MCL** en el almacén;
- **el EKF de S18 cuando las balizas no dicen cuál son**, y la verosimilitud de esas mismas balizas para MCL.

En MCL, las partículas se mueven con **la odometría que mide el robot**, con su error, como el EKF de S18.

In [ ]:
# El simulador: el de S18b, más el pasillo con partículas, el LiDAR, MCL y el EKF sin nombres. Sin detenerse.
import numpy as np
from scipy.ndimage import distance_transform_edt
from scipy.optimize import least_squares

D2R = np.pi / 180
RES, FILAS, COLS = 0.10, 90, 130            # celdas de 10 cm: 13 m en x, 9 m en y


def mapa_almacen():
    """La rejilla de ocupacion del almacen de S22 (True = ocupado). Fila = y/0,1; columna = x/0,1."""
    occ = np.zeros((FILAS, COLS), bool)
    occ[0, :] = occ[-1, :] = True              # paredes exteriores
    occ[:, 0] = occ[:, -1] = True
    occ[42:47, :] = True                       # el muro central, de y = 4,2 a 4,7 m
    occ[42:47, 8:28] = False                   #   la puerta ancha, de x = 0,8 a 2,8 m
    occ[42:47, 92:100] = False                 #   la puerta estrecha, de x = 9,2 a 10,0 m
    occ[12:36, 40:47] = True                   # estanterias de la nave de abajo
    occ[12:36, 64:71] = True
    occ[58:78, 34:41] = True                   # estanterias de la nave de arriba
    occ[58:78, 58:65] = True
    occ[58:78, 82:89] = True
    return occ


OCC = mapa_almacen()
MUELLE, PICKING = np.array([0.6, 0.6]), np.array([8.3, 8.2])
BALIZAS = np.array([[0.1, 2.0], [3.0, 0.1], [8.0, 0.1], [12.9, 2.0], [5.5, 4.15],      # nave de abajo
                    [0.1, 7.0], [4.7, 8.9], [10.5, 8.9], [12.9, 6.5], [7.0, 4.75]])    # nave de arriba
RUTA = np.array([[0.6, 0.6], [1.8, 0.6], [1.8, 5.2], [9.6, 5.2], [9.6, 0.6], [0.6, 0.6]])   # la ruta del inventario

V, HZ = 0.5, 10                              # 0,5 m/s; odometria a 10 Hz
DD = V / HZ                                  # 5 cm por paso
SIG_D, SIG_T = 0.06 / np.sqrt(20), 0.2 * D2R # por paso: 13,4 mm y 0,2° (6 cm y 0,9° por metro)
Q = np.diag([SIG_D**2, SIG_T**2])            # Q: la covarianza del ruido de proceso (la odometria)
SIG_R, SIG_PHI, ALCANCE = 0.05, 1.0 * D2R, 8.0
R = np.diag([SIG_R**2, SIG_PHI**2])          # R: la covarianza del ruido de medida (el sensor de balizas)
CADA = 10                                    # un barrido de balizas cada 10 pasos: una vez por segundo
K95 = 2.4477                                 # en el plano, la elipse de 2,45σ encierra el 95 %


def envuelve(a):
    """Un angulo en (−π, π]: sin esto, la innovacion del rumbo salta 360°."""
    return (a + np.pi) % (2 * np.pi) - np.pi


def paso(x, d):
    """El modelo de movimiento: la pose x = (x; y; θ) compuesta con la odometria d = (δ_d; δ_θ) en SE(2),
    como en S8: avanzar δ_d en la direccion del rumbo y despues girar δ_θ."""
    return np.array([x[0] + d[0] * np.cos(x[2]), x[1] + d[0] * np.sin(x[2]), envuelve(x[2] + d[1])])


def jac_paso(x, d):
    """Sus jacobianos: F_x respecto a la pose (3 x 3) y F_v respecto a la odometria (3 x 2)."""
    c, s = np.cos(x[2]), np.sin(x[2])
    Fx = np.array([[1.0, 0.0, -d[0] * s], [0.0, 1.0, d[0] * c], [0.0, 0.0, 1.0]])
    Fv = np.array([[c, 0.0], [s, 0.0], [0.0, 1.0]])
    return Fx, Fv


def h(x, b):
    """El modelo de medida: distancia y rumbo a la baliza b desde la pose x."""
    dx, dy = b[0] - x[0], b[1] - x[1]
    return np.array([np.hypot(dx, dy), envuelve(np.arctan2(dy, dx) - x[2])])


def jac_h(x, b):
    """Su jacobiano H_x respecto a la pose (2 x 3)."""
    dx, dy = b[0] - x[0], b[1] - x[1]
    r2 = dx * dx + dy * dy
    r = np.sqrt(r2)
    return np.array([[-dx / r, -dy / r, 0.0], [dy / r2, -dx / r2, -1.0]])


def libre_vista(a, b):
    """¿Ve el sensor la baliza b desde a? Sin celdas ocupadas en medio (la baliza esta en la pared)."""
    n = int(np.ceil(np.linalg.norm(b - a) / 0.02))
    p = a + np.linspace(0, 1, n)[:, None] * (b - a)
    cc = np.clip((p[:, 0] / RES).astype(int), 0, COLS - 1)
    ff = np.clip((p[:, 1] / RES).astype(int), 0, FILAS - 1)
    return not OCC[ff, cc][:-6].any()


def ruta_densa():
    """La ruta del inventario, en pasos de 5 cm."""
    pts = [RUTA[0]]
    for a, b in zip(RUTA[:-1], RUTA[1:]):
        n = int(round(np.linalg.norm(b - a) / DD))
        pts += [a + (b - a) * k / n for k in range(1, n + 1)]
    return np.array(pts)


def mandos(rd):
    """La odometria verdadera de cada paso: el avance y el giro que llevan de un punto de la ruta al siguiente."""
    d, th = [], 0.0
    for a, b in zip(rd[:-1], rd[1:]):
        thn = np.arctan2(b[1] - a[1], b[0] - a[0])
        d.append((np.linalg.norm(b - a), envuelve(thn - th)))
        th = thn
    return np.array(d)


RD = ruta_densa()
U = mandos(RD)                               # 544 pasos, 27,2 m
VERDAD = [np.array([RD[0, 0], RD[0, 1], 0.0])]
for d in U:
    VERDAD.append(paso(VERDAD[-1], d))
VERDAD = np.array(VERDAD)
VISIBLES = [[j for j, b in enumerate(BALIZAS) if np.linalg.norm(b - p[:2]) < ALCANCE and libre_vista(p[:2], b)]
            for p in VERDAD]


def simula(semilla=1):
    """Lo que leen los sensores en una pasada por la ruta: la odometria de cada paso y, una vez por segundo,
    la distancia y el rumbo de cada baliza visible (con su numero: la asociacion se supone conocida)."""
    rng = np.random.default_rng(semilla)
    odo = U + rng.normal(0, [SIG_D, SIG_T], size=U.shape)
    med = {}
    for k in range(CADA, len(VERDAD), CADA):
        med[k] = [(j, h(VERDAD[k], BALIZAS[j]) + rng.normal(0, [SIG_R, SIG_PHI])) for j in VISIBLES[k]]
    return odo, med
# =====================================================================================================
# S19 · 1. El pasillo de S18, con particulas
# =====================================================================================================
NC = 130                                     # el pasillo junto al muro, y = 5,2 m: 130 celdas de 10 cm
XC = (np.arange(NC) + 0.5) * RES             # el centro de cada celda
P_HUECO, P_FALSA = 0.8, 0.1                  # el detector de huecos de S18: p(«hueco» | puerta), p(«hueco» | pared)
AV, SIG_AV = 0.5, np.sqrt(0.2) * 0.1        # cada medida, 0,5 m mas adelante, con σ = 4,47 cm (el nucleo 0,1/0,8/0,1 de S18)


def hueco_en(x):
    """¿Esta x frente a un hueco del muro? La puerta ancha, de 0,8 a 2,8 m; la estrecha, de 9,2 a 10,0 m."""
    x = np.asarray(x)
    return ((x >= 0.8) & (x < 2.8)) | ((x >= 9.2) & (x < 10.0))


def verosimilitud_hueco(x, z):
    """p(z | x) del detector: 0,8 o 0,1 si dice «hueco»; 0,2 o 0,9 si dice «pared». Fuera del pasillo, 0."""
    p = np.where(hueco_en(x), P_HUECO if z else 1 - P_HUECO, P_FALSA if z else 1 - P_FALSA)
    return p * ((x >= 0) & (x < 13.0))


def sistematico(w, rng):
    """El remuestreo sistematico: un solo numero al azar, r, y M punteros equiespaciados (r + k)/M sobre la
    suma acumulada de los pesos. Devuelve el indice de la particula que copia cada puntero."""
    M = w.size
    punteros = (rng.random() + np.arange(M)) / M
    return np.minimum(np.searchsorted(np.cumsum(w), punteros), M - 1)


def recorre_pasillo(semilla=18, x0=0.95, n=19):
    """El recorrido del pasillo de S18 (cuaderno S18a, recorre()): el robot sale de x0 y avanza 0,5 m por medida;
    el detector dice «hueco» o «pared», con sus errores. Las mismas medidas que vio el histograma de S18."""
    rng = np.random.default_rng(semilla)
    X, Z, x = [], [], x0
    for _ in range(n):
        X.append(x)
        Z.append(bool(rng.random() < (P_HUECO if hueco_en(XC[int(x / RES)]) else P_FALSA)))
        x += AV
    return np.array(X), np.array(Z)


def mcl_pasillo(Z, M, semilla):
    """MCL en el pasillo: M particulas repartidas al azar; para cada medida, pesar y remuestrear, y despues
    mover 0,5 m con ruido. Devuelve, por medida, las particulas y sus pesos (antes de remuestrear)."""
    rng = np.random.default_rng(semilla)
    P = rng.uniform(0, 13.0, M)
    hist = []
    for z in Z:
        w = verosimilitud_hueco(P, z)                    # 2. pesar
        w = w / w.sum() if w.sum() > 0 else np.full(M, 1 / M)
        hist.append((P.copy(), w.copy()))
        P = P[sistematico(w, rng)]                       # 3. remuestrear
        P = P + AV + rng.normal(0, SIG_AV, M)            # 1. muestrear el movimiento
    return hist


def histograma_pasillo(Z):
    """El filtro de histograma de S18 con las mismas medidas: 130 celdas; avanzar 4, 5 o 6 celdas con 0,1, 0,8 y
    0,1, y lo que saldria del pasillo se pierde (cuaderno S18a, corregir() y predecir())."""
    bel = np.full(NC, 1 / NC)
    hist = []
    for z in Z:
        bel = bel * verosimilitud_hueco(XC, z)
        bel /= bel.sum()
        hist.append(bel.copy())
        nueva = np.zeros_like(bel)
        for desp, w in zip((4, 5, 6), (0.1, 0.8, 0.1)):
            nueva[desp:] += w * bel[:NC - desp]
        bel = nueva / nueva.sum()
    return hist


def masa_cerca(P, w, x, d=0.3):
    """La creencia a ±d de la x verdadera: la suma de los pesos de las particulas que caen ahi."""
    return float(np.sum(w[np.abs(P - x) <= d + 1e-9]))
# =====================================================================================================
# S19 · 2. El LiDAR de S6 en la rejilla, el campo de verosimilitud y MCL en el almacen
# =====================================================================================================
ANG = np.radians(np.arange(0, 360, 0.25))    # 1440 haces: 360° con 0,25° de resolucion
RMAX, SIG_L, P_SIN = 10.0, 0.02, 0.02        # alcance de 10 m, 2 cm de ruido, 2 % de rayos sin retorno
HACES = np.arange(0, ANG.size, 80)           # el filtro usa 18 haces, uno cada 20°
DIST = distance_transform_edt(~OCC) * RES    # el campo de verosimilitud: distancia de cada celda al obstaculo
LIBRE = DIST > 0.28                          # donde cabe el centro del robot (radio de 0,28 m, el de S22)
_FL, _CL = np.nonzero(LIBRE)


def rayos(pose, angs, occ=OCC):
    """La distancia al primer obstaculo de cada rayo, marchando a pasos de 1 cm por la rejilla."""
    t = np.arange(0.01, RMAX, 0.01)
    a = pose[2] + angs
    cc = ((pose[0] + np.outer(np.cos(a), t)) / RES).astype(int)
    ff = ((pose[1] + np.outer(np.sin(a), t)) / RES).astype(int)
    fuera = (cc < 0) | (cc >= COLS) | (ff < 0) | (ff >= FILAS)
    choca = fuera | occ[np.clip(ff, 0, FILAS - 1), np.clip(cc, 0, COLS - 1)]
    r = t[choca.argmax(axis=1)]
    r[~choca.any(axis=1)] = RMAX
    return r


def barrido(pose, rng, idx=None):
    """Un barrido del LiDAR desde la pose (los haces idx; todos si no se dan), con su ruido y sus rayos perdidos."""
    a = ANG if idx is None else ANG[idx]
    r = rayos(pose, a) + rng.normal(0, SIG_L, a.size)
    r[rng.random(a.size) < P_SIN] = RMAX
    return np.minimum(r, RMAX)


def relativa(a, b):
    """La pose de b vista desde a, en SE(2) (la ⊖ de S8): adelante, a la izquierda y el giro."""
    c, s = np.cos(a[2]), np.sin(a[2])
    d = np.asarray(b[:2]) - np.asarray(a[:2])
    return np.array([c * d[0] + s * d[1], -s * d[0] + c * d[1], envuelve(b[2] - a[2])])
def log_verosimilitud(P, r, sig=0.3, zimp=0.5, zaz=0.5):
    """El campo de verosimilitud (Thrun et al., 2005): cada haz lleva su punto final a la rejilla desde cada
    particula, y vale una campana de σ = 0,3 m sobre la distancia de ese punto a la pared mas cercana, mas un
    fondo al azar. Suma de logaritmos sobre los 18 haces; una particula dentro de una pared vale −∞."""
    a, rr = ANG[HACES], r
    ok = rr < RMAX - 1e-6                                    # los rayos sin retorno no cuentan
    a, rr = a[ok], rr[ok]
    cc = ((P[:, 0:1] + rr * np.cos(P[:, 2:3] + a)) / RES).astype(int)
    ff = ((P[:, 1:2] + rr * np.sin(P[:, 2:3] + a)) / RES).astype(int)
    dentro = (cc >= 0) & (cc < COLS) & (ff >= 0) & (ff < FILAS)
    d = np.where(dentro, DIST[np.clip(ff, 0, FILAS - 1), np.clip(cc, 0, COLS - 1)], 2.0)
    p = zimp * np.exp(-0.5 * (d / sig) ** 2) / (np.sqrt(2 * np.pi) * sig) + zaz / RMAX
    lw = np.log(p).sum(axis=1)
    c0, f0 = (P[:, 0] / RES).astype(int), (P[:, 1] / RES).astype(int)
    en = (c0 >= 0) & (c0 < COLS) & (f0 >= 0) & (f0 < FILAS)
    lw[~en] = -np.inf
    lw[en] = np.where(LIBRE[f0[en], c0[en]], lw[en], -np.inf)
    return lw


def pesos(lw):
    """De logaritmos a pesos normalizados, sin desbordar. Si todas valen −∞, pesos iguales."""
    if not np.isfinite(lw).any():
        return np.full(lw.size, 1 / lw.size)
    w = np.exp(lw - lw[np.isfinite(lw)].max())
    return w / w.sum()


def al_azar(M, rng):
    """M particulas repartidas al azar por el sitio libre del almacen, con rumbos al azar."""
    i = rng.integers(0, _FL.size, M)
    return np.column_stack([(_CL[i] + rng.random(M)) * RES, (_FL[i] + rng.random(M)) * RES,
                            rng.uniform(-np.pi, np.pi, M)])


def alrededor(pose, M, rng, s=0.10, sth=3 * D2R):
    """M particulas alrededor de una pose conocida (el seguimiento): 10 cm y 3° de σ."""
    return np.column_stack([pose[0] + rng.normal(0, s, M), pose[1] + rng.normal(0, s, M),
                            envuelve(pose[2] + rng.normal(0, sth, M))])


def mover(P, d, rng, k=2.0):
    """Muestrear el movimiento: cada particula compone la pose relativa d que mide la odometria entre dos
    actualizaciones (0,5 m) con su propio ruido, k veces el de la odometria."""
    s = k * SIG_D * np.sqrt(10)
    ax, ay = d[0] + rng.normal(0, s, len(P)), d[1] + rng.normal(0, s, len(P))
    dt = d[2] + rng.normal(0, k * SIG_T * np.sqrt(10), len(P))
    c, sn = np.cos(P[:, 2]), np.sin(P[:, 2])
    return np.column_stack([P[:, 0] + ax * c - ay * sn, P[:, 1] + ax * sn + ay * c, envuelve(P[:, 2] + dt)])


def media_circular(a, w):
    """El rumbo medio de la nube: atan2(Σ w·sen θ, Σ w·cos θ). La media de los numeros no vale cerca de ±180°."""
    return np.arctan2(np.sum(w * np.sin(a)), np.sum(w * np.cos(a)))


def mcl_almacen(k0, n, M, semilla, P0=None, secuestro=None, inyecta=0.0, sensor='lidar'):
    """MCL en el almacen: el robot recorre la ruta desde el paso k0 y el filtro se actualiza cada 10 pasos (0,5 m),
    n veces. Las particulas se mueven con la odometria que mide el robot (con su error de 6 cm y 0,9° por metro).
    secuestro = (u, k_nuevo): en la actualizacion u, el robot aparece en el paso k_nuevo de la ruta. inyecta:
    fraccion de particulas que se cambian por otras al azar en cada actualizacion. sensor: 'lidar' (18 haces y el
    campo de verosimilitud) o 'balizas' (las diez balizas, sin decir cual es cada una: como el EKF de la seccion 5).
    Devuelve, por actualizacion, la verdad, la nube, los pesos, la estimacion y su error."""
    rng = np.random.default_rng(semilla)
    odo = U + np.random.default_rng(semilla + 1_000_000).normal(0, [SIG_D, SIG_T], size=U.shape)   # lo que mide el robot
    P = al_azar(M, rng) if P0 is None else P0.copy()
    k, out = k0, []
    for u in range(n + 1):
        if secuestro and u == secuestro[0]:
            k = secuestro[1]
        verdad = VERDAD[k]
        if sensor == 'lidar':                                                   # 2. pesar
            lw = log_verosimilitud(P, barrido(verdad, rng, HACES))
        else:
            zs = [h(verdad, BALIZAS[j]) + rng.normal(0, [SIG_R, SIG_PHI]) for j in VISIBLES[k]]
            lw = log_verosimilitud_balizas(P, zs)
        w = pesos(lw)
        est = np.array([np.sum(w * P[:, 0]), np.sum(w * P[:, 1]), media_circular(P[:, 2], w)])
        out.append(dict(k=k, verdad=verdad, P=P.copy(), w=w, est=est,
                        err=float(np.hypot(*(est[:2] - verdad[:2]))),
                        err_th=float(abs(envuelve(est[2] - verdad[2]))),
                        cerca=float(np.sum(w * (np.hypot(P[:, 0] - verdad[0], P[:, 1] - verdad[1]) < 0.5)))))
        if u == n:
            break
        P = P[sistematico(w, rng)]                                            # 3. remuestrear
        if inyecta > 0:
            m = int(round(inyecta * M))
            P[rng.choice(M, m, replace=False)] = al_azar(m, rng)
        k2 = min(k + CADA, len(U))
        d = np.zeros(3)
        for dd in odo[k:k2]:
            d = paso(d, dd)                     # la pose relativa que da la odometria entre dos actualizaciones
        P = mover(P, d, rng)                                                  # 1. muestrear el movimiento
        k = k2
    return out


def localizado(e, dist=0.3, ang=5 * D2R):
    """Exito de la localizacion: la estimacion a menos de 30 cm y 5° de la verdad."""
    return e["err"] < dist and e["err_th"] < ang
# =====================================================================================================
# S19 · 3. Cuando los reflectores no se distinguen: el EKF con el vecino mas cercano
# =====================================================================================================
PUERTA_99 = 9.21                             # χ² con 2 grados de libertad: el 99 % de las medidas buenas, por debajo


def log_verosimilitud_balizas(P, zs):
    """Balizas que no dicen cual son: cada medida (distancia y rumbo) pesa una mezcla sobre las diez balizas
    (1/10 cada una). Suma de logaritmos sobre las medidas; una particula dentro de una pared vale −∞."""
    lw = np.zeros(len(P))
    for z in zs:
        acc = np.zeros(len(P))
        for b in BALIZAS:
            dx, dy = b[0] - P[:, 0], b[1] - P[:, 1]
            er = z[0] - np.hypot(dx, dy)
            ep = envuelve(z[1] - envuelve(np.arctan2(dy, dx) - P[:, 2]))
            acc += np.exp(-0.5 * ((er / SIG_R) ** 2 + (ep / SIG_PHI) ** 2))
        lw += np.log(acc / len(BALIZAS) + 1e-12)
    c0, f0 = (P[:, 0] / RES).astype(int), (P[:, 1] / RES).astype(int)
    en = (c0 >= 0) & (c0 < COLS) & (f0 >= 0) & (f0 < FILAS)
    lw[~en] = -np.inf
    lw[en] = np.where(LIBRE[f0[en], c0[en]], lw[en], -np.inf)
    return lw


def simula_sin_nombre(semilla=1, k0=0):
    """Como simula(), desde el paso k0, pero el sensor no dice que baliza es: solo la distancia y el rumbo."""
    rng = np.random.default_rng(semilla)
    odo = U[k0:] + rng.normal(0, [SIG_D, SIG_T], size=U[k0:].shape)
    med = {}
    for k in range(k0 + CADA, len(VERDAD), CADA):
        med[k] = [h(VERDAD[k], BALIZAS[j]) + rng.normal(0, [SIG_R, SIG_PHI]) for j in VISIBLES[k]]
    return odo, med


def ekf_vecino(odo, med, x0, Sigma0, k0=0, puerta=PUERTA_99):
    """El EKF de S18 cuando no se sabe que baliza es: cada medida se atribuye a la baliza del mapa con menor
    d² = νᵀ·S⁻¹·ν desde la estimacion (el vecino mas cercano), y se descarta si pasa de la puerta."""
    x, S_ = np.array(x0, float), Sigma0.copy()
    X, P = [x.copy()], [S_.copy()]
    for k, d in enumerate(odo, start=k0 + 1):
        Fx, Fv = jac_paso(x, d)
        x = paso(x, d)
        S_ = Fx @ S_ @ Fx.T + Fv @ Q @ Fv.T
        for z in med.get(k, []):
            mejor = None
            for b in BALIZAS:
                Hx = jac_h(x, b)
                nu = z - h(x, b); nu[1] = envuelve(nu[1])
                Sm = Hx @ S_ @ Hx.T + R
                d2 = float(nu @ np.linalg.solve(Sm, nu))
                if mejor is None or d2 < mejor[0]:
                    mejor = (d2, Hx, nu, Sm)
            d2, Hx, nu, Sm = mejor
            if puerta is not None and d2 > puerta:
                continue
            L = S_ @ Hx.T @ np.linalg.inv(Sm)
            x = x + L @ nu; x[2] = envuelve(x[2])
            S_ = (np.eye(3) - L @ Hx) @ S_
        X.append(x.copy()); P.append(S_.copy())
    return np.array(X), np.array(P)

MAPA = ListedColormap(['white', '#4A4F63'])

def dibuja_almacen(ax, alfa=1.0):
    ax.imshow(OCC, cmap=MAPA, origin='lower', extent=(0, 13, 0, 9), interpolation='nearest', alpha=alfa)
    ax.set_xlim(0, 13); ax.set_ylim(0, 9); ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])

print(f'Almacén de {COLS*RES:.0f} x {FILAS*RES:.0f} m ; sitio libre para el centro del robot: {LIBRE.sum()} celdas ; ruta de {len(U)} pasos')

## 1. El pasillo de S18, con partículas

*En casa.* Es lo de las diapositivas 68 a 70.

**Una partícula es una hipótesis**: un sitio posible del robot. Una nube de M partículas aproxima la creencia: la probabilidad de estar en una zona es la fracción de partículas que caen en ella. Con una creencia de dos picos, la nube tiene dos grupos; una gaussiana no podría.

El filtro de partículas es **el filtro de Bayes de S18 con muestras**, en tres pasos:

1. **Muestrear** (predecir): cada partícula avanza lo que dice la odometría, con su propio ruido.
2. **Pesar** (corregir): su peso es la verosimilitud p(z | x) de la medida desde ella, normalizado; es la puerta de S18.
3. **Remuestrear**: se sortean M nuevas, cada una con probabilidad igual a su peso. El **sorteo sistemático** usa un solo número al azar r y M punteros equiespaciados, (r + k)/M, sobre la suma acumulada de los pesos.

Primero, las cinco partículas de la diapositiva 69, a mano.

In [ ]:
x = np.array([1.2, 1.9, 4.0, 9.1, 11.0])
x = x + 0.5 + np.array([0.03, -0.02, 0.04, 0.01, -0.03])     # 1. muestrear: avanzan 0,5 m, cada una con su ruido
w = verosimilitud_hueco(x, True)                            # 2. pesar: el detector dice «hueco»
print('tras muestrear:', '; '.join(coma(v) for v in x))
print('p(z | x):', '; '.join(coma(v, 1) for v in w), ' suma', coma(w.sum(), 1))
w = w / w.sum()
print('pesos:', '; '.join(coma(v, 3) for v in w))
acum = np.cumsum(w)
punteros = (0.5 + np.arange(5)) / 5                         # 3. remuestrear, sistemático, con r = 0,5
copia = np.searchsorted(acum, punteros)
print('acumulada:', '; '.join(coma(v, 3) for v in acum), ' punteros:', '; '.join(coma(v, 1) for v in punteros))
print('copian la', ', '.join(str(i + 1) for i in copia), '->', '; '.join(coma(v) for v in x[copia]))

Ahora, **MCL en el pasillo** (la localización de Monte Carlo: el filtro de partículas aplicado a localizarse) con **las mismas medidas que vio el histograma de S18** (semilla 18, desde x = 0,95 m), y el histograma de S18 al lado.

In [ ]:
X, Z = recorre_pasillo()                                     # las medidas de S18
H = histograma_pasillo(Z)
PF = mcl_pasillo(Z, 200, 1)
print('medidas:', ' '.join('H' if z else 'p' for z in Z))
fig, axs = plt.subplots(4, 1, figsize=(10, 7), sharex=True)
for ax, i in zip(axs, (0, 3, 14, 18)):
    P, w = PF[i]
    hh, ed = np.histogram(P, bins=130, range=(0, 13), weights=w)
    ax.bar(ed[:-1] + 0.05, hh, width=0.1, color=IQS_AZUL, alpha=0.85, label='200 partículas')
    ax.plot(XC, H[i], color=AMBAR, lw=1.6, label='histograma de S18')
    ax.axvline(X[i], color=ROJO, ls='--')
    for a, b in ((0.8, 2.8), (9.2, 10.0)):
        ax.axvspan(a, b, color=IQS_VERDE, alpha=0.12)
    ax.set_yticks([])
    ax.set_title(f"x = {coma(X[i])} m, «{'hueco' if Z[i] else 'pared'}»: a ±0,3 m, {coma(100*masa_cerca(P, w, X[i]), 0)} %"
                 f" (histograma: {coma(100*H[i][np.abs(XC - X[i]) <= 0.3 + 1e-9].sum(), 0)} %)", fontsize=10, loc='left')
axs[0].legend(fontsize=9); axs[-1].set_xlabel('x en el pasillo [m]'); ejes(axs[-1])
plt.tight_layout(); plt.show()

In [ ]:
# Cuántas partículas: 100 recorridos, cada uno con sus medidas; ¿cuántos acaban con más del 80 % a ±0,3 m?
t0 = time.time()
for M in (20, 50, 200, 1000):
    ok = 0
    for sem in range(100):
        X_, Z_ = recorre_pasillo(sem)
        P, w = mcl_pasillo(Z_, M, 1000 + sem)[-1]
        ok += masa_cerca(P, w, X_[-1]) > 0.8
    print(f'M = {M:4d}: {ok} de 100')
okh = sum(histograma_pasillo(recorre_pasillo(s)[1])[-1][np.abs(XC - recorre_pasillo(s)[0][-1]) <= 0.3 + 1e-9].sum() > 0.8 for s in range(100))
print(f'el histograma de S18: {okh} de 100 ({coma(time.time() - t0, 1)} s)')

Con pocas partículas decide el azar; con muchas, la nube hace lo que el histograma (39, 54, 64 y 66 de 100, frente a 68). Con mil partículas fallan 34 recorridos: los 32 en que falla también el histograma, porque el detector se equivoca, y 2 más, del propio filtro.

## 2. El LiDAR y el campo de verosimilitud

*En clase, sin detenerte.* Es lo de la diapositiva 71.

El LiDAR de S6 da 1440 distancias. Para pesar una partícula con él se usa el **campo de verosimilitud** (Thrun et al., 2005): cada haz lleva su punto final al mapa desde la partícula, y vale más cuanto más cerca cae de una pared, con una campana de 0,3 m sobre esa distancia y un fondo al azar para los haces que no cuadran (la mitad del peso para cada cosa). El peso de la partícula es la suma de los logaritmos de 18 haces, uno cada 20°. La distancia de cada celda a la pared más cercana se calcula una vez, con una transformada de distancia: es `DIST`.

In [ ]:
pose = VERDAD[54]
r = barrido(pose, np.random.default_rng(0))
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
dibuja_almacen(a1)
a = pose[2] + ANG
a1.plot(pose[0] + r*np.cos(a), pose[1] + r*np.sin(a), '.', ms=1.5, color=ROJO)
a1.plot(*pose[:2], 'o', color=ROJO); a1.set_title('un barrido del LiDAR desde el paso 54 de la ruta', fontsize=10)
im = a2.imshow(DIST, origin='lower', extent=(0, 13, 0, 9), cmap='viridis_r', vmax=1.5)
a2.set_title('el campo: distancia a la pared más cercana [m]', fontsize=10); a2.set_xticks([]); a2.set_yticks([])
plt.colorbar(im, ax=a2, fraction=0.03, format=COMA); plt.tight_layout(); plt.show()
P = np.array([pose, pose + [0.3, 0, 0], pose + [0, 0, 0.2], [10.5, 2.5, 0.0]])
print('log w de la pose verdadera, desplazada 30 cm, girada 11° y en otro sitio libre, (10,5; 2,5):',
      '; '.join(coma(v, 1) for v in log_verosimilitud(P, r[HACES])))

Cuanto más se aparta la pose, menos vale: −6,5 en la verdadera, −9,0 desplazada 30 cm, −21,1 girada 11° y −29,3 en otro sitio. Una partícula dentro de una pared o de una estantería vale −∞ (peso 0).

## 3. La nube en el almacén

*En clase, sin detenerte.* Diez mil partículas repartidas al azar por el sitio libre, con rumbos al azar; el robot sube por la nave de abajo. El filtro se actualiza cada medio metro: cada partícula se mueve con la odometría que mide el robot (con el doble de su ruido), se pesa con el LiDAR y se remuestrea.

El **rumbo medio** de la nube se calcula con la **media circular**, atan2(Σ w·sen θ, Σ w·cos θ): cerca de ±180°, la media de los números no vale.

In [ ]:
t0 = time.time()
nube = mcl_almacen(54, 10, 10000, 222)
print(f'{coma((time.time() - t0)/11*1000, 0)} ms por actualización')
fig, axs = plt.subplots(1, 4, figsize=(15, 3.2))
for ax, u in zip(axs, (0, 1, 3, 7)):
    e = nube[u]
    dibuja_almacen(ax)
    i = np.random.default_rng(u).choice(len(e['P']), 3000, replace=False)
    ax.scatter(e['P'][i, 0], e['P'][i, 1], s=1.5, color=IQS_AZUL, alpha=0.4)
    ax.plot(*e['verdad'][:2], 'o', color=ROJO, ms=7)
    cerca = np.mean(np.hypot(e['P'][:, 0] - e['verdad'][0], e['P'][:, 1] - e['verdad'][1]) < 1)
    ax.set_title(f'actualización {u}: {coma(100*cerca, 0)} % a menos de 1 m', fontsize=9)
plt.tight_layout(); plt.show()
for u in (0, 3, 7):
    e = nube[u]
    print(f"actualización {u}: estimación ({coma(e['est'][0])}; {coma(e['est'][1])}; {coma(e['est'][2]*180/np.pi, 1)}°), a {coma(e['err']*100, 1)} cm")
oeste = mcl_almacen(372, 8, 10000, 226)[-1]
print('hacia el oeste: media de los números', coma(np.sum(oeste['w'] * oeste['P'][:, 2])*180/np.pi, 1), '° ; media circular',
      coma(media_circular(oeste['P'][:, 2], oeste['w'])*180/np.pi, 1), '°')

Antes de que la nube se junte, la media pesada no significa nada: en la actualización 0 cae a varios metros del robot. Después, el error baja a unos centímetros.

## 4. Taller: cuántas hacen falta

*En clase.* Veinte arranques en puntos al azar de la ruta, sin saber dónde, con tres mil partículas y con cinco mil. Localizado: a los 8 m (16 actualizaciones), la estimación a menos de 30 cm y 5° del robot. **¿Cuántos se localizan? La tabla de la diapositiva 72 usa cincuenta arranques, y estos veinte son sus veinte primeros: ¿cuánto se mueve la tasa con veinte?** Esta celda tarda unos cinco segundos.

In [ ]:
arranques = np.random.default_rng(11).integers(0, len(VERDAD) - 170, 50)[:20]
for M in (3000, 5000):
    t0 = time.time()
    exitos = [localizado(mcl_almacen(int(k0), 16, M, 200 + i)[-1]) for i, k0 in enumerate(arranques)]
    print(f'M = {M:5d}: se localizan {sum(exitos)} de 20 ({coma(time.time() - t0, 1)} s)')

Con 3000 partículas se localizan 8 de 20, el 40 %; con los cincuenta arranques de la diapositiva 72, 30 de 50, el 60 %. **Una tasa medida con N arranques tiene su propia incertidumbre**, de unos √(p·(1 − p)/N): con p = 0,6, ±0,11 con N = 20 y ±0,07 con N = 50. Para la entrega 3, cuantos más arranques, mejor medida.

Los que fallan son el **empobrecimiento**: si ninguna partícula cae cerca del robot, el remuestreo solo copia las que hay, y la nube se queda en otro sitio, segura de sí. Con los cincuenta arranques de la diapositiva 72: 18 de 50 con 1000, 30 con 3000 y 47 con 10 000. **AMCL**, el localizador de Nav2 (S22), adapta el número: muchas partículas mientras la nube está repartida y pocas cuando se junta. La tasa de éxito en N arranques es la métrica de la localización global para la entrega 3.

## 5. Taller: el EKF que arranca en el hueco equivocado

*En clase.* El robot está en el pasillo, frente a la puerta estrecha; el EKF de S18 arranca frente a la ancha, con σ = 50 cm a lo largo: la joroba más alta del pasillo tras el primer «hueco». Las balizas **no dicen cuál son**: cada medida se atribuye a la baliza que mejor cuadra desde donde el EKF cree estar. **¿Dónde acaba el EKF, y cuánto se fía de sí mismo?** Y después, MCL desde cero en el mismo sitio y **con las mismas balizas sin nombre**: cada medida pesa una mezcla sobre las diez balizas (`log_verosimilitud_balizas`), así que solo cambia el filtro.

In [ ]:
k0 = 266
odo, med = simula_sin_nombre(3, k0)
verdad = VERDAD[k0:]                             # el robot, desde k0
Sigma0 = np.diag([0.5**2, 0.05**2, (2*D2R)**2])
for puerta in (None, PUERTA_99):
    Xe, Pe = ekf_vecino(odo, med, (1.9, 5.2, 0.0), Sigma0, k0=k0, puerta=puerta)
    err = np.hypot(*(Xe[-1, :2] - verdad[-1, :2]))
    sigma = np.sqrt(np.linalg.eigvalsh(Pe[-1][:2, :2]).max())
    print(f"{'sin puerta' if puerta is None else 'con la puerta de 9,21'}: acaba a {coma(err)} m del robot con σ = {coma(sigma*100, 1)} cm ({coma(err/sigma, 0)} σ)")
nube = mcl_almacen(k0, 10, 10000, 30, sensor='balizas')
print(f"MCL con las mismas balizas sin nombre, desde cero: en la actualización 1, a {coma(nube[1]['err']*100, 1)} cm ; al final, a {coma(nube[-1]['err']*100, 1)} cm")
Xe, Pe = ekf_vecino(odo, med, (1.9, 5.2, 0.0), Sigma0, k0=k0, puerta=None)
fig, ax = plt.subplots(figsize=(7, 5))
dibuja_almacen(ax)
ax.plot(verdad[:, 0], verdad[:, 1], color=ROJO, lw=2, label='el robot'); ax.plot(Xe[:, 0], Xe[:, 1], color=IQS_AZUL, lw=2, label='lo que cree el EKF')
ax.legend(fontsize=9); plt.show()

Sin puerta, el EKF se queda con la baliza que mejor le cuadra desde donde cree estar, se convence y **encoge su σ**: acaba a 8,5 m del robot con σ = 4,5 cm, a 187 σ. Su covarianza no mira la medida: encoge igual aunque la asociación sea falsa. Con la puerta de Mahalanobis de S18 rechaza casi todo: avisa, pero no sabe dónde está. La nube, desde cero y con las mismas balizas sin nombre, está a 2 cm en la actualización 1: el mismo sensor, otro filtro.

## 6. Taller: el robot secuestrado

*En clase.* El robot sale del muelle localizado y, en la actualización 10, alguien lo lleva de la puerta ancha a la estrecha. **¿Se recupera la nube sin partículas nuevas? ¿Y cambiando en cada actualización un 5 % de la nube por partículas al azar? ¿Basta un 2 %?** Es la semilla de la figura de la diapositiva 73.

In [ ]:
P0 = alrededor(VERDAD[0], 10000, np.random.default_rng(0))
fig, ax = plt.subplots(figsize=(9, 3.5))
for iny, color in ((0.0, ROJO), (0.05, IQS_VERDE), (0.02, AMBAR)):
    r = mcl_almacen(0, 30, 10000, 400, P0=P0, secuestro=(10, 330), inyecta=iny)
    ax.plot([e['err'] for e in r], '-o', ms=3, color=color, label=f'{coma(100*iny, 0)} % nuevas')
    ok = [localizado(e) for e in r]
    desde = next((u for u in range(10, len(ok)) if all(ok[u:])), None)
    print(f'inyección {coma(100*iny, 0)} %: error 1, 3 y 10 actualizaciones después:', ', '.join(coma(r[u]['err'], 2) for u in (11, 13, 20)), 'm ;',
          'no se recupera' if desde is None else f'localizada desde la actualización {desde}')
ax.axvline(10, color='k', ls=':'); ax.set_xlabel('actualización (cada 0,5 m)'); ax.set_ylabel('error [m]'); ejes(ax); ax.legend()
plt.tight_layout(); plt.show()

Sin partículas nuevas no queda ninguna cerca del robot, y el remuestreo no puede crearla: en veinte pruebas, no se recupera ninguna. Con un 5 % nuevas en cada actualización se recuperan 18 de 20, en 4 actualizaciones de mediana; con un 2 %, 16 de 20, en 6,5 (aquí, desde la actualización 15, frente a la 11 con el 5 %): basta, pero tarda. Inyectar tiene precio: ese 5 % de la nube siempre está perdido. AMCL puede inyectarlas solo cuando cae la verosimilitud media, que es como se nota un secuestro: es el ejercicio 3.

### Ejercicio 1 (en casa): el detector de la puerta en el pasillo

En la sección 1, cambia el detector de huecos por el de la puerta estrecha de S18: `P_HUECO, P_FALSA = 0.6, 0.2`. ¿Cuántos de los 100 recorridos se localizan con 200 partículas y con 1000? ¿Y con el histograma? ¿Lo arreglan las partículas?

In [ ]:
# Ejercicio 1
# P_HUECO, P_FALSA = 0.6, 0.2
# ... repite la celda de los 100 recorridos ...
# P_HUECO, P_FALSA = 0.8, 0.1      # y déjalo como estaba

### Ejercicio 2 (en casa): pocas partículas y muchas nuevas

Con los 20 arranques de la sección 4, prueba M = 1000 y M = 300, sin partículas nuevas y con un 10 % nuevas en cada actualización (`inyecta=0.10`). ¿Compensa la inyección la falta de partículas?

In [ ]:
# Ejercicio 2
# for M, iny in ((1000, 0.0), (1000, 0.10), (300, 0.0), (300, 0.10)):
#     exitos = [localizado(mcl_almacen(int(k0), 16, M, 200 + i, inyecta=iny)[-1]) for i, k0 in enumerate(arranques)]

### Ejercicio 3 (en casa): inyectar solo cuando hace falta

Copia el bucle de `mcl_almacen` y cambia la inyección fija por la adaptativa (Thrun et al., 2005): en cada actualización, la verosimilitud media de la nube, `media` (la media de `exp(lw/18)`), alimenta dos medias que la siguen, una lenta y otra rápida, `w_lenta += 0.05*(media - w_lenta)` y `w_rapida += 0.5*(media - w_rapida)`, y se inyecta la fracción `max(0, 1 - w_rapida/w_lenta)`. Con el secuestro de la sección 6 (veinte semillas, de la 400 a la 419), ¿cuántas se recuperan, y cuánto inyecta antes y justo después del secuestro?

In [ ]:
# Ejercicio 3
# w_lenta = w_rapida = None
# ... en cada actualización:
#     media = np.mean(np.exp(lw / HACES.size)[np.isfinite(lw)])
#     w_lenta = media if w_lenta is None else w_lenta + 0.05*(media - w_lenta)
#     w_rapida = media if w_rapida is None else w_rapida + 0.5*(media - w_rapida)
#     fraccion = max(0.0, 1 - w_rapida / w_lenta)

---

## Soluciones

**Ejercicio 1.** Con el detector de 0,6 y 0,2 se localizan 5 de 100 con 200 partículas y 2 con 1000; el histograma, 5 (como en el ejercicio 2 del S18a). Las partículas no lo arreglan: el filtro es el mismo, y el problema es el sensor. Ningún número de partículas saca información de donde no la hay.

**Ejercicio 2.** De los 20 arranques: con 1000 partículas, 7 sin inyectar y 12 con un 10 % nuevas; con 300, 8 y 9 (sin inyectar, 300 da uno más que 1000: con veinte arranques la tasa se mueve, como en la sección 4). La inyección ayuda a encontrar el sitio si la nube tiene partículas suficientes para seguirlo cuando lo encuentra; con 300 apenas ayuda. Por eso AMCL sube el número de partículas mientras la nube está repartida.

**Ejercicio 3.** Se recuperan las 20, en 3 actualizaciones de mediana. Antes del secuestro inyecta de media el 3,6 % (la verosimilitud media apenas se mueve); justo después, hasta el 52 % (mediana de los máximos de cada prueba). Es la versión práctica de la inyección: casi nada en marcha normal, mucho cuando la nube deja de explicar lo que ve el LiDAR.

---

## Para llevarse de esta sesión

- **Una nube de partículas es una creencia con tantos picos como haga falta**: la probabilidad de una zona es la fracción de partículas que caen en ella.
- **El filtro de partículas es el filtro de Bayes de S18 con muestras**: muestrear (predecir), pesar con la verosimilitud (corregir y normalizar) y remuestrear.
- **La localización global necesita muchas partículas**: con pocas, la nube se empobrece y se queda en otro sitio; la métrica es la tasa de éxito en N arranques. AMCL adapta el número.
- **Sin puerta, el EKF equivocado no avisa**: con una sola gaussiana, se queda con la hipótesis que tiene y encoge su σ; con la puerta, avisa, pero no sabe dónde está.
- **Sin partículas nuevas no hay recuperación de un secuestro**: el remuestreo copia, no crea.
- **Los rumbos se promedian con la media circular**, no número a número.

*Cuaderno del curso 82514 Mecatrónica y Robótica · IQS Universitat Ramon Llull · curso 2026/27*

*© Guillermo Reyes Carmenaty · Publicado bajo [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es): puedes usarlo, adaptarlo y redistribuirlo, incluso con fines comerciales, siempre que reconozcas la autoría.*